In [1]:
import argparse
import os
import time
import shutil

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.backends.cudnn as cudnn

import torchvision
from torchvision import datasets
import torchvision.transforms as transforms
from IPython.display import Markdown, clear_output, display
from models import *

# number of subprocesses to use for data loading
num_workers = 0
# how many samples per batch to load
batch_size = 128

global best_prec
if torch.cuda.is_available():
    device = torch.device('cuda')
    cudnn.benchmark = True       # fixed 32x32 input -> let cuDNN pick the fastest conv kernels
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print('device:', device)

# same CIFAR-10 pipeline as HW3, where the VGG16 checkpoint was trained
normalize = transforms.Normalize(mean=[0.491, 0.482, 0.447], std=[0.247, 0.243, 0.262])

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    normalize,
])
test_transform = transforms.Compose([
    transforms.ToTensor(),
    normalize,
])

# choose the training and test datasets
train_data = datasets.CIFAR10(root='data', train=True, download=True,  transform=train_transform)
test_data  = datasets.CIFAR10(root='data', train=False, download=True, transform=test_transform)

# prepare data loaders
train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True,
    num_workers=num_workers)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=batch_size, shuffle=False,
    num_workers=num_workers)

PATH = "result/vgg16/model_best.pth.tar"


def load_step1_model():
    """Return a fresh copy of the trained HW3 VGG16. Every case starts from this checkpoint."""
    new_model = VGG16()
    checkpoint = torch.load(PATH, map_location=device)
    new_model.load_state_dict(checkpoint['state_dict'])
    return new_model.to(device)


model = load_step1_model()



def train(gamma=0, lr=0.01, weight_decay=1e-4, n_epochs=50):
    criterion = nn.CrossEntropyLoss().to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)

    # per epoch, all the training data set is used once
    model.train() # prep model for training


    for epoch in range(n_epochs):
        # monitor training loss
        train_loss = 0.0
        train_loss1 = 0.0
        train_loss2 = 0.0
        
        ###################
        # train the model #
        ###################
        for data, target in train_loader:
            data, target = data.to(device), target.to(device) # loading to GPU
            # clear the gradients of all optimized variables
            optimizer.zero_grad()
            
            # forward pass: compute predicted outputs by passing inputs to the model
            output = model(data)

            # calculate the loss
            loss1 = criterion(output, target)               # cross entropy loss
            loss2 = model.features[0].weight.abs().sum()    # SUM(|Weights|) of the first conv layer only

            loss = loss1 + gamma*loss2
            
            loss.backward()
            optimizer.step()
            train_loss1 += loss1.item()*data.size(0) # as loss is tensor, .item() needed to get the value
            train_loss2 += loss2.item()*data.size(0) # as loss is tensor, .item() needed to get the value
            train_loss  += loss.item()*data.size(0) # as loss is tensor, .item() needed to get the value
            
        # print training statistics 
        # calculate average loss over an epoch
        train_loss1 = train_loss1/len(train_loader.dataset)
        train_loss2 = train_loss2/len(train_loader.dataset)
        train_loss = train_loss/len(train_loader.dataset)

        print('Epoch: {} \tTraining Loss1: {:.6f}'.format(epoch+1, train_loss1))
        print('Epoch: {} \tTraining Loss2: {:.6f}'.format(epoch+1, train_loss2))
        print('Epoch: {} \tTraining Loss: {:.6f}'.format(epoch+1, train_loss))
        
    # see following link for details of state_dict   
    # https://pytorch.org/tutorials/recipes/recipes/saving_and_loading_a_general_checkpoint.html
    #torch.save({
    #            'epoch': epoch,
    #            'model_state_dict': model.state_dict(),
    #            'optimizer_state_dict': optimizer.state_dict(),
    #            'loss': train_loss,
    #            }, PATH)


def evaluate():
    """Return (test accuracy in %, conv1 weight abs sum) of the current model."""
    model.eval()
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device) # move to device
            output = model(data)
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()
    accuracy = 100. * correct / len(test_loader.dataset)
    return accuracy, model.features[0].weight.abs().sum().item()


def fine_tune(g):
    """Train from the step 1 checkpoint with gamma = g, save the model, and return evaluate().

    A model that is already saved is reloaded instead of retrained, so the notebook can be
    re-run to build the PDF. If you change any training setting, delete
    result/vgg16/gamma_*.pth first, or the old models will be reused.
    """
    global model
    save_path = 'result/vgg16/gamma_{}.pth'.format(g)
    model = load_step1_model()
    if os.path.exists(save_path):
        model.load_state_dict(torch.load(save_path, map_location=device))
    else:
        train(gamma=g)
        torch.save(model.state_dict(), save_path)
        clear_output()    # training logs must not appear in the PDF
    return evaluate()


results = {}    # gamma -> (test accuracy in %, conv1 weight abs sum)


def report(g):
    accuracy, abs_sum = results[g]
    print('gamma = {}: test accuracy = {:.2f}%, conv1 weight abs sum = {:.2f}'.format(g, accuracy, abs_sum))

device: cuda
Files already downloaded and verified
Files already downloaded and verified


## 1) Initial Version VGG16

In [2]:
model = load_step1_model()
results[0] = evaluate()
report(0)

gamma = 0: test accuracy = 92.80%, conv1 weight abs sum = 241.63


## 2) Adding new loss term

In [3]:
results[1] = fine_tune(1)
report(1)

gamma = 1: test accuracy = 81.27%, conv1 weight abs sum = 20.56


## 3) After fine-tuning the new loss term

In [4]:
gammas = [0.005, 0.05, 0.5]

for g in gammas:
    results[g] = fine_tune(g)

for g in gammas:
    report(g)

# optimal gamma: smallest conv1 abs sum among the runs that stay above 80% accuracy
# (set best_gamma by hand if another run gives a better balance)
candidates = [g for g in results if g != 0 and results[g][0] > 80]
best_gamma = min(candidates, key=lambda g: results[g][1])

print('\nOptimal gamma = {}'.format(best_gamma))
print('Test accuracy = {:.2f}%, conv1 weight abs sum = {:.2f}'.format(*results[best_gamma]))
print('Cost function: loss = loss1 + {} * loss2'.format(best_gamma))
print('  loss1 = CrossEntropyLoss, loss2 = SUM(|W_conv1|)')

gamma = 0.005: test accuracy = 89.77%, conv1 weight abs sum = 8.09
gamma = 0.05: test accuracy = 80.90%, conv1 weight abs sum = 6.11
gamma = 0.5: test accuracy = 81.89%, conv1 weight abs sum = 12.82

Optimal gamma = 0.05
Test accuracy = 80.90%, conv1 weight abs sum = 6.11
Cost function: loss = loss1 + 0.05 * loss2
  loss1 = CrossEntropyLoss, loss2 = SUM(|W_conv1|)


## 4) Summary

In [5]:
rows = [
    ('1. Initial version', 'loss1', 0),
    ('2. New loss term (γ = 1)', 'loss1 + loss2', 1),
    ('3. Fine-tuned (optimal γ)', 'loss1 + γ × loss2 (γ = {})'.format(best_gamma), best_gamma),
]

table = '| Case | Cost function | Test accuracy (%) | Conv1 weight abs sum |\n|---|---|---|---|\n'
for case, cost, g in rows:
    table += '| {} | {} | {:.2f} | {:.2f} |\n'.format(case, cost, results[g][0], results[g][1])

display(Markdown(table))

| Case | Cost function | Test accuracy (%) | Conv1 weight abs sum |
|---|---|---|---|
| 1. Initial version | loss1 | 92.80 | 241.63 |
| 2. New loss term (γ = 1) | loss1 + loss2 | 81.27 | 20.56 |
| 3. Fine-tuned (optimal γ) | loss1 + γ × loss2 (γ = 0.05) | 80.90 | 6.11 |


### Discussion of results

Adding the conv1 weight absolute sum times a gamma term lowered the overall accuracy of the model, dropping from above 90% to the low 80% range. The gamma value chosen for the joint/customized loss function directly affected the accuracy of the model overall, and also the weights' absolute sum. The smallest gamma = 0.005 chosen dropped the weight sum down drastically while maintaining near 90% accuracy. We see that choosing 0.05, an order of magnitude higher, slightly reduced the weight absolute sum further, but reduced the accuracy of the model even more. Large gamma values chosen maintained near the same accuracy, but the absolute sum of the weights increased back again. 

In [6]:
# every gamma that was tried
table = '| γ | Test accuracy (%) | Conv1 weight abs sum |\n|---|---|---|\n'
for g in sorted(results):
    table += '| {} | {:.2f} | {:.2f} |\n'.format(g, results[g][0], results[g][1])

display(Markdown(table))

| γ | Test accuracy (%) | Conv1 weight abs sum |
|---|---|---|
| 0 | 92.80 | 241.63 |
| 0.005 | 89.77 | 8.09 |
| 0.05 | 80.90 | 6.11 |
| 0.5 | 81.89 | 12.82 |
| 1 | 81.27 | 20.56 |
